## **Parte IV**
Implementação das tarefas 17 a 20

---


In [1]:
import os
import time

import psycopg2
from psycopg2 import sql, errors
import getpass

In [11]:
## Criando o Database
DB_SUPER = "postgres"
DB_USER = "jaidezardin"
DB_HOST = os.getenv("PGHOST", "localhost")
DB_PORT = int(os.getenv("PGPORT", 5432))

NEW_DB = "reservas"
NEW_TABLE = "Assentos"

conn = psycopg2.connect(dbname=DB_SUPER, user=DB_USER, host=DB_HOST, port=DB_PORT)
conn.autocommit = True
cur = conn.cursor()
try:
    cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(NEW_DB)))
    print(f"Banco de dados {NEW_DB} criado.")
except errors.DuplicateDatabase:
    print(f"Banco de dados {NEW_DB} já existe.")
finally:
    cur.close()
    conn.close()

Banco de dados reservas já existe.


In [12]:
## Criando a tabela

conn = psycopg2.connect(dbname=NEW_DB, user=DB_USER, host=DB_HOST, port=DB_PORT)
cur = conn.cursor()

cur.execute(sql.SQL(f"""
CREATE TABLE {NEW_TABLE} (
num_voo integer,
disp bool,
primary key (num_voo),
check (num_voo > 0),
check (num_voo <= 200)
)
""")
)

conn.commit()
cur.close()
conn.close()


In [13]:
## Transações
import random

def reserva_version_a(conn, isolation_level="READ COMMITED"):
    cur = conn.cursor()
    try:
        cur.execute(f"set transaction isolation level {isolation_level}")
        cur.execute("select num_voo from Assentos where disp = true")
        open = cur.fetchall()

        if not open:
            conn.rollback()
            return None

        time.sleep(1)
        chosen = random.choice(open)[0]

        cur.execute(
            """
            update assentos set disp = false where num_voo = %s and disp = true
            """,
            (chosen,)
        )

        if cur.rowcount == 0:
            conn.rollback()
            return None

        conn.commit()
        return chosen
    except Exception as e:
        conn.rollback()
        print(f"Erro na reserva versão A: {e}")
        return None
    finally:
        cur.close()


def reserva_version_b(conn, isolation_level="READ COMMITED"):
    cur = conn.cursor()
    try:
        cur.execute(f"set transaction isolation level {isolation_level}")
        cur.execute("select num_voo from Assentos where disp = true")
        open = cur.fetchall()
        conn.commit()

        if not open:
            conn.rollback()
            return None

        time.sleep(1)
        chosen = random.choice(open)[0]

        cur.execute(f"set transaction isolation level {isolation_level}")
        cur.execute (
            "update Assentos set disp = false where num_voo = %s and disp = true",
            (chosen,)
        )
        if cur.rowcount == 0:
            conn.rollback()
            return None
        conn.commit()
        return chosen
    except Exception as e:
        conn.rollback()
        print("Erro na reserva versão B: ", e)
        return None
    finally:
        cur.close()


In [ ]:
class MainExecution:
    def __init__(self, reserva_version, k, isolation_level="READ COMMITED"):
        self.reserva_version = reserva_version
        self.k = k
        self.isolation_level = isolation_level

    def setup_database(self):
        conn = psycopg2.connect(dbname=NEW_DB, user=DB_USER, host=DB_HOST, port=DB_PORT)
        cur = conn.cursor()
        cur.execute(sql.SQL(f"truncate table {NEW_TABLE}"))
        cur.close()
        conn.close()
